# CIFAR-10 이미지 분류 CNN 모델 — 최신 PyTorch 코드

이 노트북은 CIFAR-10 이미지 데이터를 사용하여 CNN 기반 이미지 분류 모델을 학습하고 평가하는 예제입니다.

구성은 다음 순서로 되어 있습니다.

1. 실행 환경 확인
2. 라이브러리 불러오기
3. 하이퍼파라미터 설정
4. CIFAR-10 데이터셋 준비
5. 샘플 이미지 시각화
6. CNN 모델 설계
7. 손실함수와 최적화 알고리즘 설정
8. 모델 학습
9. 모델 평가
10. 혼동행렬과 클래스별 정확도 확인
11. 예측 결과 시각화


In [ ]:
# 현재 노트북에서 사용할 주요 라이브러리를 불러옵니다.
# torch는 PyTorch의 핵심 패키지로, 텐서 연산과 딥러닝 모델 학습에 사용됩니다.
import torch

# torch.nn은 신경망 계층, 활성화 함수, 손실함수 등을 제공하는 모듈입니다.
import torch.nn as nn

# torch.optim은 SGD, Adam, AdamW 같은 최적화 알고리즘을 제공하는 모듈입니다.
import torch.optim as optim

# DataLoader는 데이터셋을 미니배치 단위로 나누어 모델에 공급하는 도구입니다.
from torch.utils.data import DataLoader

# torchvision은 이미지 데이터셋, 이미지 변환, 사전학습 모델 등을 제공하는 패키지입니다.
import torchvision

# torchvision.transforms는 이미지 데이터를 텐서로 바꾸거나 정규화하는 전처리 기능을 제공합니다.
import torchvision.transforms as transforms

# torchvision.datasets.CIFAR10은 CIFAR-10 데이터셋을 쉽게 내려받고 사용할 수 있게 해줍니다.
from torchvision.datasets import CIFAR10

# numpy는 혼동행렬 계산 등 수치 연산에 사용됩니다.
import numpy as np

# matplotlib.pyplot은 이미지와 그래프를 출력하는 데 사용됩니다.
import matplotlib.pyplot as plt

# time은 학습 시간 측정에 사용됩니다.
from time import time

# 운영체제별 DataLoader 설정을 안전하게 처리하기 위해 os 모듈을 사용합니다.
import os

# PyTorch 버전을 출력하여 현재 실행 환경을 확인합니다.
print("PyTorch version:", torch.__version__)

# CUDA 사용 가능 여부를 확인합니다.
print("CUDA available:", torch.cuda.is_available())

## 1. 하이퍼파라미터 설정

하이퍼파라미터는 모델이 학습되기 전에 사람이 직접 정하는 값입니다.  
대표적으로 학습 횟수, 배치 크기, 학습률 등이 있습니다.


In [ ]:
# 재현 가능한 실험을 위해 난수 시드를 고정합니다.
# 같은 코드를 다시 실행했을 때 가능한 한 비슷한 결과가 나오도록 도와줍니다.
SEED = 111

# PyTorch의 CPU 난수 생성 시드를 고정합니다.
torch.manual_seed(SEED)

# GPU가 있을 경우 GPU 난수 생성 시드도 고정합니다.
torch.cuda.manual_seed_all(SEED)

# 에포크는 전체 학습 데이터를 몇 번 반복해서 학습할지 정하는 값입니다.
# 처음 실습할 때는 2~5 정도로 작게 두고, 성능 개선 실험에서는 더 크게 늘릴 수 있습니다.
EPOCHS = 5

# 배치 크기는 한 번의 학습 단계에서 모델에 넣는 이미지 개수입니다.
# CIFAR-10은 이미지가 작기 때문에 64 또는 128을 자주 사용합니다.
BATCH_SIZE = 64

# 학습률은 가중치를 한 번 업데이트할 때 얼마나 크게 움직일지 정하는 값입니다.
# 너무 크면 학습이 불안정하고, 너무 작으면 학습이 느려질 수 있습니다.
LEARNING_RATE = 0.001

# CIFAR-10 데이터셋을 저장할 폴더 경로입니다.
DATA_ROOT = "./cifar"

# GPU가 사용 가능하면 cuda를 사용하고, 그렇지 않으면 cpu를 사용합니다.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 현재 모델 학습에 사용할 장치를 출력합니다.
print("사용 장치:", device)

# Windows 또는 Colab 환경에서 안정적으로 동작하도록 num_workers를 설정합니다.
# Windows에서 문제가 생기면 0을 사용하는 것이 가장 안전합니다.
NUM_WORKERS = 2 if os.name != "nt" else 0

# DataLoader가 GPU로 데이터를 더 빠르게 옮길 수 있도록 pin_memory 설정을 지정합니다.
PIN_MEMORY = True if torch.cuda.is_available() else False

## 2. 데이터 준비

CIFAR-10은 32×32 크기의 컬러 이미지 데이터셋입니다.  
총 10개의 클래스로 구성되어 있으며, 각 이미지는 RGB 3채널을 가집니다.

클래스는 다음과 같습니다.

`airplane`, `automobile`, `bird`, `cat`, `deer`, `dog`, `frog`, `horse`, `ship`, `truck`


In [ ]:
# CIFAR-10 데이터의 클래스 이름을 리스트로 저장합니다.
# 모델의 숫자 예측값을 사람이 읽을 수 있는 이름으로 바꿀 때 사용합니다.
classes = [
    "airplane",    # 0번 클래스: 비행기
    "automobile",  # 1번 클래스: 자동차
    "bird",        # 2번 클래스: 새
    "cat",         # 3번 클래스: 고양이
    "deer",        # 4번 클래스: 사슴
    "dog",         # 5번 클래스: 개
    "frog",        # 6번 클래스: 개구리
    "horse",       # 7번 클래스: 말
    "ship",        # 8번 클래스: 배
    "truck"        # 9번 클래스: 트럭
]

# CIFAR-10 이미지는 RGB 3채널 이미지이므로 채널별 평균값을 지정합니다.
# 여기서는 각 채널을 [-1, 1] 범위로 변환하기 위해 평균 0.5를 사용합니다.
mean = [0.5, 0.5, 0.5]

# CIFAR-10 이미지는 RGB 3채널 이미지이므로 채널별 표준편차를 지정합니다.
# 표준편차 0.5를 사용하면 ToTensor()로 [0, 1]이 된 값이 Normalize 후 [-1, 1] 근처가 됩니다.
std = [0.5, 0.5, 0.5]

# 학습 데이터에 적용할 이미지 전처리 규칙을 정의합니다.
train_transform = transforms.Compose([
    # 이미지를 좌우로 무작위 반전하여 데이터 다양성을 늘립니다.
    transforms.RandomHorizontalFlip(),

    # 이미지를 4픽셀 패딩한 뒤 32x32 크기로 무작위 잘라내어 위치 변화에 강한 모델을 만듭니다.
    transforms.RandomCrop(32, padding=4),

    # PIL 이미지 또는 NumPy 이미지를 PyTorch 텐서 형태로 변환합니다.
    # 변환 후 이미지 모양은 [채널, 높이, 너비]가 됩니다.
    transforms.ToTensor(),

    # 이미지 픽셀 값을 평균과 표준편차를 기준으로 정규화합니다.
    transforms.Normalize(mean=mean, std=std)
])

# 평가 데이터에 적용할 이미지 전처리 규칙을 정의합니다.
# 평가에서는 무작위 변형을 넣지 않아야 공정한 평가가 가능합니다.
test_transform = transforms.Compose([
    # 이미지를 PyTorch 텐서로 변환합니다.
    transforms.ToTensor(),

    # 학습 데이터와 동일한 방식으로 정규화합니다.
    transforms.Normalize(mean=mean, std=std)
])

# CIFAR-10 학습 데이터셋을 생성합니다.
train_dataset = CIFAR10(
    root=DATA_ROOT,          # 데이터셋 저장 위치를 지정합니다.
    train=True,              # 학습용 데이터셋을 사용합니다.
    download=True,           # 데이터셋이 없으면 자동으로 다운로드합니다.
    transform=train_transform # 학습용 전처리 규칙을 적용합니다.
)

# CIFAR-10 평가 데이터셋을 생성합니다.
test_dataset = CIFAR10(
    root=DATA_ROOT,         # 데이터셋 저장 위치를 지정합니다.
    train=False,            # 평가용 데이터셋을 사용합니다.
    download=True,          # 데이터셋이 없으면 자동으로 다운로드합니다.
    transform=test_transform # 평가용 전처리 규칙을 적용합니다.
)

# 학습 데이터셋을 미니배치 단위로 공급하는 DataLoader를 생성합니다.
train_loader = DataLoader(
    dataset=train_dataset,     # 사용할 학습 데이터셋을 지정합니다.
    batch_size=BATCH_SIZE,     # 한 배치에 포함할 이미지 개수를 지정합니다.
    shuffle=True,              # 학습 데이터 순서를 매 에포크마다 섞습니다.
    num_workers=NUM_WORKERS,   # 데이터를 불러올 작업자 프로세스 수를 지정합니다.
    pin_memory=PIN_MEMORY      # GPU 사용 시 데이터 전송 효율을 높입니다.
)

# 평가 데이터셋을 미니배치 단위로 공급하는 DataLoader를 생성합니다.
test_loader = DataLoader(
    dataset=test_dataset,      # 사용할 평가 데이터셋을 지정합니다.
    batch_size=BATCH_SIZE,     # 한 배치에 포함할 이미지 개수를 지정합니다.
    shuffle=False,             # 평가는 순서를 섞을 필요가 없습니다.
    num_workers=NUM_WORKERS,   # 데이터를 불러올 작업자 프로세스 수를 지정합니다.
    pin_memory=PIN_MEMORY      # GPU 사용 시 데이터 전송 효율을 높입니다.
)

# 학습 데이터 개수를 출력합니다.
print("학습 데이터 개수:", len(train_dataset))

# 평가 데이터 개수를 출력합니다.
print("평가 데이터 개수:", len(test_dataset))

## 3. 샘플 이미지 확인

정규화된 이미지를 화면에 보기 위해 다시 `[0, 1]` 범위로 되돌린 뒤 출력합니다.


In [ ]:
# 정규화된 이미지를 화면에 출력하기 위한 함수입니다.
def imshow(img):
    # Normalize에서 x_new = (x - 0.5) / 0.5로 바꾸었으므로,
    # 시각화를 위해 img = img * 0.5 + 0.5로 원래 [0, 1] 범위에 가깝게 되돌립니다.
    img = img * 0.5 + 0.5

    # 텐서를 NumPy 배열로 변환합니다.
    np_img = img.numpy()

    # PyTorch 이미지는 [채널, 높이, 너비] 순서입니다.
    # matplotlib은 [높이, 너비, 채널] 순서를 요구하므로 축 순서를 바꿉니다.
    np_img = np.transpose(np_img, (1, 2, 0))

    # 이미지 값을 안전하게 [0, 1] 범위로 제한합니다.
    np_img = np.clip(np_img, 0, 1)

    # 이미지를 화면에 출력합니다.
    plt.imshow(np_img)

    # 축 눈금은 이미지 분류 실습에서 필요하지 않으므로 숨깁니다.
    plt.axis("off")

    # 그래프를 화면에 표시합니다.
    plt.show()

# 학습 데이터에서 첫 번째 미니배치를 가져옵니다.
dataiter = iter(train_loader)

# 최신 Python 반복자 방식으로 다음 배치를 가져옵니다.
images, labels = next(dataiter)

# 여러 이미지를 격자 형태의 하나의 이미지로 합칩니다.
grid_img = torchvision.utils.make_grid(images[:16], nrow=4)

# 합쳐진 이미지를 출력합니다.
imshow(grid_img)

# 이미지에 해당하는 정답 라벨 이름을 출력합니다.
print("정답 라벨:", [classes[label] for label in labels[:16]])

## 4. 모델 설계 상세 설명

이 노트북의 모델은 CNN 구조를 사용합니다. CNN은 이미지 분류에서 자주 사용되는 신경망 구조입니다.

### 4.1 입력층

CIFAR-10 이미지는 `3 × 32 × 32` 형태입니다.

- `3`: RGB 색상 채널
- `32`: 이미지 높이
- `32`: 이미지 너비

모델은 한 번에 여러 장의 이미지를 배치로 받기 때문에 실제 입력 모양은 다음과 같습니다.

`[배치크기, 3, 32, 32]`

### 4.2 합성곱 계층

합성곱 계층은 이미지의 지역적인 특징을 추출합니다.  
초기 계층은 선, 모서리, 색상 변화 같은 단순한 특징을 찾고, 뒤쪽 계층은 더 복잡한 패턴을 찾습니다.

이 모델에서는 세 개의 합성곱 블록을 사용합니다.

첫 번째 블록은 RGB 이미지에서 32개의 특징맵을 추출합니다.  
두 번째 블록은 32개의 특징맵을 64개로 확장합니다.  
세 번째 블록은 64개의 특징맵을 128개로 확장합니다.

### 4.3 활성화 함수

각 합성곱 뒤에는 `ReLU`를 사용합니다.  
ReLU는 음수 값을 0으로 바꾸고 양수 값은 그대로 통과시킵니다.  
이 과정은 모델이 단순한 선형 계산이 아니라 복잡한 패턴을 학습할 수 있게 해줍니다.

### 4.4 배치 정규화

`BatchNorm2d`는 각 계층의 출력 분포를 안정적으로 만들어 학습을 더 빠르고 안정적으로 진행하게 합니다.

### 4.5 풀링 계층

`MaxPool2d`는 특징맵의 크기를 줄이면서 중요한 특징을 남깁니다.  
이미지 크기가 줄어들기 때문에 계산량도 줄어듭니다.

### 4.6 Dropout

`Dropout`은 학습 중 일부 뉴런을 무작위로 꺼서 모델이 특정 뉴런에 과도하게 의존하지 않도록 합니다.  
이는 과적합을 줄이는 데 도움이 됩니다.

### 4.7 출력층

마지막 출력층은 10개의 값을 출력합니다.  
CIFAR-10은 10개의 클래스를 가지므로 출력 뉴런도 10개입니다.

중요한 점은 출력층에 `Softmax`를 직접 넣지 않는다는 것입니다.  
`CrossEntropyLoss`는 내부적으로 `LogSoftmax`와 `NLLLoss` 계산을 함께 처리하므로, 모델은 정규화되지 않은 점수인 `logits`를 그대로 출력하는 것이 올바른 방식입니다.


In [ ]:
# CIFAR-10 분류를 위한 CNN 모델 클래스를 정의합니다.
class CIFAR10CNN(nn.Module):
    # 모델에서 사용할 계층을 초기화합니다.
    def __init__(self, num_classes=10):
        # 부모 클래스 nn.Module의 초기화 기능을 실행합니다.
        super().__init__()

        # 첫 번째 합성곱 블록을 정의합니다.
        self.features = nn.Sequential(
            # 입력 채널 3개 RGB 이미지를 받아 32개의 특징맵을 생성합니다.
            nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1),

            # 32개 특징맵의 분포를 안정화하여 학습을 돕습니다.
            nn.BatchNorm2d(32),

            # 음수 값은 0으로 만들고 양수 값은 그대로 통과시켜 비선형성을 추가합니다.
            nn.ReLU(inplace=True),

            # 32x32 특징맵을 16x16으로 줄입니다.
            nn.MaxPool2d(kernel_size=2, stride=2),

            # 두 번째 합성곱 계층으로 32개 특징맵을 64개 특징맵으로 확장합니다.
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),

            # 64개 특징맵의 분포를 안정화합니다.
            nn.BatchNorm2d(64),

            # 두 번째 합성곱 결과에 비선형성을 추가합니다.
            nn.ReLU(inplace=True),

            # 16x16 특징맵을 8x8로 줄입니다.
            nn.MaxPool2d(kernel_size=2, stride=2),

            # 세 번째 합성곱 계층으로 64개 특징맵을 128개 특징맵으로 확장합니다.
            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1),

            # 128개 특징맵의 분포를 안정화합니다.
            nn.BatchNorm2d(128),

            # 세 번째 합성곱 결과에 비선형성을 추가합니다.
            nn.ReLU(inplace=True),

            # 8x8 특징맵을 4x4로 줄입니다.
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        # 분류기 부분을 정의합니다.
        self.classifier = nn.Sequential(
            # 128개의 4x4 특징맵을 1차원 벡터로 펼칩니다.
            nn.Flatten(),

            # 펼친 벡터 128*4*4를 256차원 은닉층으로 변환합니다.
            nn.Linear(in_features=128 * 4 * 4, out_features=256),

            # 은닉층 출력에 비선형성을 추가합니다.
            nn.ReLU(inplace=True),

            # 학습 중 일부 뉴런을 무작위로 비활성화하여 과적합을 줄입니다.
            nn.Dropout(p=0.3),

            # 256차원 은닉 표현을 10개 클래스 점수로 변환합니다.
            # 여기서는 Softmax를 사용하지 않고 logits를 그대로 출력합니다.
            nn.Linear(in_features=256, out_features=num_classes)
        )

    # 입력 이미지가 모델을 통과하는 순전파 과정을 정의합니다.
    def forward(self, x):
        # 입력 이미지를 합성곱 특징 추출기에 통과시킵니다.
        x = self.features(x)

        # 추출된 특징을 분류기에 통과시켜 클래스별 점수를 계산합니다.
        x = self.classifier(x)

        # 최종 클래스별 logits를 반환합니다.
        return x

# 모델 객체를 생성합니다.
model = CIFAR10CNN(num_classes=10)

# 모델을 GPU 또는 CPU 장치로 이동합니다.
model = model.to(device)

# 모델 구조를 출력합니다.
print(model)

# 더미 입력을 만들어 출력 모양을 확인합니다.
dummy_input = torch.randn(1, 3, 32, 32).to(device)

# 그래디언트 계산 없이 모델 출력 모양만 확인합니다.
with torch.no_grad():
    # 더미 입력을 모델에 넣어 출력값을 계산합니다.
    dummy_output = model(dummy_input)

# 출력 텐서의 모양을 확인합니다.
print("모델 출력 모양:", dummy_output.shape)

## 5. 손실함수와 최적화 알고리즘 상세 설명

### 5.1 손실함수: CrossEntropyLoss

이 모델은 다중 클래스 분류 문제를 해결합니다.  
CIFAR-10은 10개의 클래스 중 하나를 맞히는 문제이므로 `CrossEntropyLoss`를 사용합니다.

`CrossEntropyLoss`는 모델이 출력한 클래스별 점수와 실제 정답 라벨을 비교하여 손실값을 계산합니다.  
손실값이 작을수록 모델의 예측이 정답에 가깝다는 뜻입니다.

모델의 출력은 다음과 같은 형태입니다.

`[배치크기, 클래스개수]`

예를 들어 배치 크기가 64이고 클래스가 10개라면 출력 모양은 다음과 같습니다.

`[64, 10]`

정답 라벨은 다음과 같은 형태입니다.

`[배치크기]`

예를 들어 정답 라벨은 `[3, 1, 9, 0, ...]`처럼 각 이미지의 정답 클래스 번호를 가집니다.

`CrossEntropyLoss`를 사용할 때는 모델 마지막에 `Softmax`를 넣지 않는 것이 일반적입니다.  
그 이유는 `CrossEntropyLoss`가 내부적으로 클래스 점수를 확률 형태로 변환하는 계산을 포함하고 있기 때문입니다.

### 5.2 최적화 알고리즘: AdamW

최적화 알고리즘은 손실값을 줄이기 위해 모델의 가중치를 어떻게 수정할지 결정합니다.  
이 노트북에서는 최신 PyTorch 실무 코드에서 자주 사용하는 `AdamW`를 사용합니다.

`AdamW`는 Adam의 장점을 유지하면서 가중치 감쇠를 더 올바르게 적용하는 방식입니다.  
가중치 감쇠는 모델의 가중치가 지나치게 커지는 것을 막아 과적합을 줄이는 데 도움을 줍니다.

### 5.3 학습 과정 요약

학습은 다음 순서로 진행됩니다.

1. 이미지를 모델에 입력합니다.
2. 모델이 10개 클래스에 대한 점수를 출력합니다.
3. 손실함수가 예측값과 정답값을 비교하여 손실을 계산합니다.
4. `optimizer.zero_grad()`로 이전 기울기를 초기화합니다.
5. `loss.backward()`로 역전파를 수행하여 기울기를 계산합니다.
6. `optimizer.step()`으로 모델 가중치를 업데이트합니다.


In [ ]:
# 다중 클래스 분류 문제에 적합한 손실함수를 생성합니다.
criterion = nn.CrossEntropyLoss()

# AdamW 최적화 알고리즘을 생성합니다.
optimizer = optim.AdamW(
    model.parameters(),       # 학습할 모델 파라미터를 최적화 대상에 넣습니다.
    lr=LEARNING_RATE,         # 학습률을 지정합니다.
    weight_decay=1e-4         # 과적합 완화를 위한 가중치 감쇠를 지정합니다.
)

# 학습률 스케줄러를 생성합니다.
# 에포크가 진행될수록 학습률을 조금씩 줄여 더 안정적으로 수렴하도록 합니다.
scheduler = optim.lr_scheduler.StepLR(
    optimizer,     # 학습률을 조절할 최적화 객체를 지정합니다.
    step_size=3,   # 3 에포크마다 학습률을 줄입니다.
    gamma=0.5      # 학습률에 0.5를 곱해 절반으로 줄입니다.
)

## 6. 학습 함수와 평가 함수 정의

학습 코드를 함수로 나누면 코드가 깔끔해지고 재사용하기 쉬워집니다.


In [ ]:
# 한 에포크 동안 모델을 학습하는 함수를 정의합니다.
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    # 모델을 학습 모드로 전환합니다.
    # Dropout과 BatchNorm은 학습 모드와 평가 모드에서 동작 방식이 다릅니다.
    model.train()

    # 에포크 전체 손실 합계를 저장할 변수를 초기화합니다.
    running_loss = 0.0

    # 정답을 맞힌 이미지 개수를 저장할 변수를 초기화합니다.
    correct = 0

    # 전체 이미지 개수를 저장할 변수를 초기화합니다.
    total = 0

    # DataLoader에서 미니배치를 하나씩 가져옵니다.
    for batch_idx, (images, labels) in enumerate(dataloader):
        # 입력 이미지를 학습 장치로 이동합니다.
        images = images.to(device)

        # 정답 라벨을 학습 장치로 이동합니다.
        labels = labels.to(device)

        # 이전 배치에서 계산된 기울기를 초기화합니다.
        optimizer.zero_grad(set_to_none=True)

        # 이미지를 모델에 넣어 클래스별 logits를 계산합니다.
        outputs = model(images)

        # 모델 출력과 정답 라벨을 비교하여 손실값을 계산합니다.
        loss = criterion(outputs, labels)

        # 손실값을 기준으로 역전파를 수행하여 각 파라미터의 기울기를 계산합니다.
        loss.backward()

        # 계산된 기울기를 사용하여 모델 파라미터를 업데이트합니다.
        optimizer.step()

        # 현재 배치의 손실값에 배치 크기를 곱해 누적합니다.
        running_loss += loss.item() * images.size(0)

        # 클래스별 점수 중 가장 큰 값의 인덱스를 예측 클래스로 선택합니다.
        preds = outputs.argmax(dim=1)

        # 예측값과 정답값이 같은 개수를 누적합니다.
        correct += (preds == labels).sum().item()

        # 현재 배치의 이미지 개수를 전체 개수에 누적합니다.
        total += labels.size(0)

        # 일정 간격으로 배치 손실을 출력합니다.
        if (batch_idx + 1) % 200 == 0:
            # 현재 배치 번호와 손실값을 출력합니다.
            print(f"  batch {batch_idx + 1:04d}/{len(dataloader):04d} | loss: {loss.item():.4f}")

    # 에포크 평균 손실을 계산합니다.
    epoch_loss = running_loss / total

    # 에포크 평균 정확도를 계산합니다.
    epoch_acc = correct / total

    # 평균 손실과 평균 정확도를 반환합니다.
    return epoch_loss, epoch_acc


# 모델을 평가하는 함수를 정의합니다.
def evaluate(model, dataloader, criterion, device):
    # 모델을 평가 모드로 전환합니다.
    model.eval()

    # 평가 전체 손실 합계를 저장할 변수를 초기화합니다.
    running_loss = 0.0

    # 정답을 맞힌 이미지 개수를 저장할 변수를 초기화합니다.
    correct = 0

    # 전체 이미지 개수를 저장할 변수를 초기화합니다.
    total = 0

    # 예측값을 저장할 리스트를 초기화합니다.
    all_preds = []

    # 정답값을 저장할 리스트를 초기화합니다.
    all_labels = []

    # 평가에서는 기울기 계산이 필요 없으므로 no_grad를 사용합니다.
    with torch.no_grad():
        # DataLoader에서 미니배치를 하나씩 가져옵니다.
        for images, labels in dataloader:
            # 입력 이미지를 평가 장치로 이동합니다.
            images = images.to(device)

            # 정답 라벨을 평가 장치로 이동합니다.
            labels = labels.to(device)

            # 이미지를 모델에 넣어 클래스별 logits를 계산합니다.
            outputs = model(images)

            # 모델 출력과 정답 라벨을 비교하여 손실값을 계산합니다.
            loss = criterion(outputs, labels)

            # 현재 배치의 손실값에 배치 크기를 곱해 누적합니다.
            running_loss += loss.item() * images.size(0)

            # 클래스별 점수 중 가장 큰 값의 인덱스를 예측 클래스로 선택합니다.
            preds = outputs.argmax(dim=1)

            # 예측값과 정답값이 같은 개수를 누적합니다.
            correct += (preds == labels).sum().item()

            # 현재 배치의 이미지 개수를 전체 개수에 누적합니다.
            total += labels.size(0)

            # CPU로 이동한 예측값을 리스트에 저장합니다.
            all_preds.append(preds.cpu())

            # CPU로 이동한 정답값을 리스트에 저장합니다.
            all_labels.append(labels.cpu())

    # 평가 평균 손실을 계산합니다.
    epoch_loss = running_loss / total

    # 평가 평균 정확도를 계산합니다.
    epoch_acc = correct / total

    # 모든 배치의 예측값을 하나의 텐서로 합칩니다.
    all_preds = torch.cat(all_preds)

    # 모든 배치의 정답값을 하나의 텐서로 합칩니다.
    all_labels = torch.cat(all_labels)

    # 평균 손실, 평균 정확도, 전체 예측값, 전체 정답값을 반환합니다.
    return epoch_loss, epoch_acc, all_preds, all_labels

## 7. 모델 학습

각 에포크마다 학습 손실, 학습 정확도, 평가 손실, 평가 정확도를 출력합니다.


In [ ]:
# 학습 기록을 저장할 딕셔너리를 생성합니다.
history = {
    "train_loss": [], # 에포크별 학습 손실을 저장합니다.
    "train_acc": [],  # 에포크별 학습 정확도를 저장합니다.
    "test_loss": [],  # 에포크별 평가 손실을 저장합니다.
    "test_acc": []    # 에포크별 평가 정확도를 저장합니다.
}

# 학습 시작 시간을 기록합니다.
start_time = time()

# 지정한 에포크 수만큼 반복합니다.
for epoch in range(EPOCHS):
    # 현재 에포크 번호를 출력합니다.
    print(f"\nEpoch {epoch + 1}/{EPOCHS}")

    # 한 에포크 동안 모델을 학습하고 학습 손실과 정확도를 받습니다.
    train_loss, train_acc = train_one_epoch(
        model=model,             # 학습할 모델을 전달합니다.
        dataloader=train_loader,  # 학습용 DataLoader를 전달합니다.
        criterion=criterion,      # 손실함수를 전달합니다.
        optimizer=optimizer,      # 최적화 알고리즘을 전달합니다.
        device=device             # 학습 장치를 전달합니다.
    )

    # 평가 데이터로 모델 성능을 측정합니다.
    test_loss, test_acc, _, _ = evaluate(
        model=model,            # 평가할 모델을 전달합니다.
        dataloader=test_loader, # 평가용 DataLoader를 전달합니다.
        criterion=criterion,    # 손실함수를 전달합니다.
        device=device           # 평가 장치를 전달합니다.
    )

    # 학습률 스케줄러를 한 단계 진행합니다.
    scheduler.step()

    # 학습 손실을 기록합니다.
    history["train_loss"].append(train_loss)

    # 학습 정확도를 기록합니다.
    history["train_acc"].append(train_acc)

    # 평가 손실을 기록합니다.
    history["test_loss"].append(test_loss)

    # 평가 정확도를 기록합니다.
    history["test_acc"].append(test_acc)

    # 현재 에포크의 결과를 출력합니다.
    print(
        f"train_loss: {train_loss:.4f} | "
        f"train_acc: {train_acc * 100:.2f}% | "
        f"test_loss: {test_loss:.4f} | "
        f"test_acc: {test_acc * 100:.2f}%"
    )

# 학습 종료 시간을 기록합니다.
end_time = time()

# 전체 학습 시간을 출력합니다.
print(f"\n총 학습 시간: {end_time - start_time:.1f}초")

## 8. 학습 과정 시각화

손실 그래프와 정확도 그래프를 확인하면 모델이 제대로 학습되고 있는지 파악할 수 있습니다.


In [ ]:
# 그래프의 x축으로 사용할 에포크 번호 리스트를 생성합니다.
epochs_range = range(1, EPOCHS + 1)

# 손실 그래프를 그릴 그림을 생성합니다.
plt.figure(figsize=(8, 5))

# 학습 손실 그래프를 그립니다.
plt.plot(epochs_range, history["train_loss"], marker="o", label="Train Loss")

# 평가 손실 그래프를 그립니다.
plt.plot(epochs_range, history["test_loss"], marker="o", label="Test Loss")

# 그래프 제목을 지정합니다.
plt.title("Loss Curve")

# x축 이름을 지정합니다.
plt.xlabel("Epoch")

# y축 이름을 지정합니다.
plt.ylabel("Loss")

# 범례를 표시합니다.
plt.legend()

# 격자를 표시합니다.
plt.grid(True)

# 그래프를 출력합니다.
plt.show()

# 정확도 그래프를 그릴 그림을 생성합니다.
plt.figure(figsize=(8, 5))

# 학습 정확도 그래프를 그립니다.
plt.plot(epochs_range, history["train_acc"], marker="o", label="Train Accuracy")

# 평가 정확도 그래프를 그립니다.
plt.plot(epochs_range, history["test_acc"], marker="o", label="Test Accuracy")

# 그래프 제목을 지정합니다.
plt.title("Accuracy Curve")

# x축 이름을 지정합니다.
plt.xlabel("Epoch")

# y축 이름을 지정합니다.
plt.ylabel("Accuracy")

# 범례를 표시합니다.
plt.legend()

# 격자를 표시합니다.
plt.grid(True)

# 그래프를 출력합니다.
plt.show()

## 9. 최종 평가와 혼동행렬

혼동행렬은 어떤 클래스를 어떤 클래스로 잘못 예측했는지 확인할 수 있는 표입니다.

행은 실제 정답 클래스이고, 열은 모델이 예측한 클래스입니다.


In [ ]:
# 평가 데이터 전체에 대해 최종 손실, 정확도, 예측값, 정답값을 계산합니다.
final_loss, final_acc, all_preds, all_labels = evaluate(
    model=model,            # 평가할 모델을 전달합니다.
    dataloader=test_loader, # 평가용 DataLoader를 전달합니다.
    criterion=criterion,    # 손실함수를 전달합니다.
    device=device           # 평가 장치를 전달합니다.
)

# 최종 평가 손실을 출력합니다.
print(f"최종 평가 손실: {final_loss:.4f}")

# 최종 평가 정확도를 출력합니다.
print(f"최종 평가 정확도: {final_acc * 100:.2f}%")

# 10x10 혼동행렬을 0으로 초기화합니다.
confusion = np.zeros((10, 10), dtype=int)

# 전체 정답값과 예측값을 하나씩 비교합니다.
for true_label, pred_label in zip(all_labels.numpy(), all_preds.numpy()):
    # 행은 실제 정답 클래스, 열은 예측 클래스로 누적합니다.
    confusion[true_label, pred_label] += 1

# 혼동행렬을 출력합니다.
print("\n혼동행렬:")
print(confusion)

# 클래스별 정확도를 출력합니다.
print("\n클래스별 정확도:")
for i, class_name in enumerate(classes):
    # 해당 클래스의 전체 정답 개수를 계산합니다.
    class_total = confusion[i].sum()

    # 해당 클래스에서 정확히 맞힌 개수를 계산합니다.
    class_correct = confusion[i, i]

    # 0으로 나누는 상황을 피하기 위해 class_total이 0보다 클 때만 계산합니다.
    class_acc = class_correct / class_total * 100 if class_total > 0 else 0

    # 클래스 이름과 정확도를 출력합니다.
    print(f"{class_name:10s}: {class_acc:6.2f}%")

In [ ]:
# 혼동행렬을 이미지로 시각화합니다.
plt.figure(figsize=(9, 8))

# 혼동행렬 값을 색상 이미지로 표시합니다.
plt.imshow(confusion, interpolation="nearest")

# 그래프 제목을 지정합니다.
plt.title("Confusion Matrix")

# 색상 막대를 표시합니다.
plt.colorbar()

# x축 눈금 위치를 생성합니다.
tick_marks = np.arange(len(classes))

# x축에 예측 클래스 이름을 표시합니다.
plt.xticks(tick_marks, classes, rotation=45)

# y축에 실제 클래스 이름을 표시합니다.
plt.yticks(tick_marks, classes)

# y축 이름을 지정합니다.
plt.ylabel("True Label")

# x축 이름을 지정합니다.
plt.xlabel("Predicted Label")

# 각 칸에 숫자를 표시합니다.
for i in range(confusion.shape[0]):
    # 각 열을 반복합니다.
    for j in range(confusion.shape[1]):
        # 해당 칸의 값을 텍스트로 표시합니다.
        plt.text(j, i, str(confusion[i, j]), ha="center", va="center")

# 그래프가 잘리지 않도록 레이아웃을 조정합니다.
plt.tight_layout()

# 그래프를 출력합니다.
plt.show()

## 10. 예측 결과 확인

평가 이미지 일부를 가져와서 모델의 예측 결과와 실제 정답을 함께 확인합니다.


In [ ]:
# 평가 데이터에서 첫 번째 미니배치를 가져옵니다.
test_iter = iter(test_loader)

# 최신 Python 반복자 방식으로 다음 배치를 가져옵니다.
test_images, test_labels = next(test_iter)

# 평가 이미지를 모델 장치로 이동합니다.
test_images_device = test_images.to(device)

# 모델을 평가 모드로 전환합니다.
model.eval()

# 예측할 때는 기울기 계산이 필요 없으므로 no_grad를 사용합니다.
with torch.no_grad():
    # 평가 이미지를 모델에 넣어 클래스별 logits를 계산합니다.
    test_outputs = model(test_images_device)

    # 클래스별 logits 중 가장 큰 값의 인덱스를 예측 클래스로 선택합니다.
    test_preds = test_outputs.argmax(dim=1).cpu()

# 출력할 이미지 개수를 지정합니다.
num_show = 8

# 지정한 개수만큼 이미지를 출력할 그림을 생성합니다.
plt.figure(figsize=(14, 4))

# 이미지 개수만큼 반복합니다.
for idx in range(num_show):
    # 여러 이미지를 한 줄로 배치하기 위한 subplot을 생성합니다.
    plt.subplot(1, num_show, idx + 1)

    # 정규화된 이미지를 [0, 1] 범위로 되돌립니다.
    img = test_images[idx] * 0.5 + 0.5

    # PyTorch 이미지 텐서를 matplotlib 형식으로 변환합니다.
    img = np.transpose(img.numpy(), (1, 2, 0))

    # 이미지 픽셀 값을 안전하게 [0, 1] 범위로 제한합니다.
    img = np.clip(img, 0, 1)

    # 이미지를 출력합니다.
    plt.imshow(img)

    # 예측 클래스 이름을 가져옵니다.
    pred_name = classes[test_preds[idx].item()]

    # 실제 정답 클래스 이름을 가져옵니다.
    true_name = classes[test_labels[idx].item()]

    # 제목에 예측값과 정답값을 함께 표시합니다.
    plt.title(f"P:{pred_name}\nT:{true_name}", fontsize=9)

    # 축 눈금은 숨깁니다.
    plt.axis("off")

# 그래프가 겹치지 않도록 레이아웃을 조정합니다.
plt.tight_layout()

# 이미지를 출력합니다.
plt.show()

## 11. 모델 저장과 불러오기

학습이 끝난 모델은 `state_dict` 형태로 저장하는 것이 일반적입니다.  
나중에 같은 모델 구조를 만든 뒤 저장된 가중치를 불러와 다시 사용할 수 있습니다.


In [ ]:
# 모델 가중치를 저장할 파일 이름을 지정합니다.
MODEL_PATH = "cifar10_cnn_adamw.pth"

# 모델의 학습된 가중치만 저장합니다.
torch.save(model.state_dict(), MODEL_PATH)

# 저장 완료 메시지를 출력합니다.
print("모델 가중치 저장 완료:", MODEL_PATH)

# 같은 구조의 새 모델 객체를 생성합니다.
loaded_model = CIFAR10CNN(num_classes=10)

# 저장된 가중치를 새 모델에 불러옵니다.
loaded_model.load_state_dict(torch.load(MODEL_PATH, map_location=device))

# 새 모델을 학습 장치로 이동합니다.
loaded_model = loaded_model.to(device)

# 불러온 모델을 평가 모드로 전환합니다.
loaded_model.eval()

# 불러오기 완료 메시지를 출력합니다.
print("모델 가중치 불러오기 완료")

## 12. 성능 개선 실험 방향

정확도를 더 높이려면 다음 항목을 실험할 수 있습니다.

1. `EPOCHS`를 20 이상으로 늘립니다.
2. `BATCH_SIZE`를 128로 변경해 봅니다.
3. 합성곱 계층 수를 더 늘립니다.
4. `AdamW`의 `LEARNING_RATE`를 `0.0005`, `0.0003` 등으로 조정합니다.
5. `RandomRotation`, `ColorJitter` 같은 데이터 증강을 추가합니다.
6. ResNet 같은 사전 정의 CNN 구조를 사용합니다.

단, 모델이 복잡해질수록 학습 시간이 늘어나므로 GPU 사용을 권장합니다.
